# FinGuard ID — Validasi Data dengan Great Expectations (0.18.19) · Revisi 4

Notebook ini memvalidasi **output task `transform`** (`data/finguard_transformed.csv`) sebelum data dikirim ke Elasticsearch. Environment yang dipakai sama dengan Milestone 3 (`h8_env`, GX 0.18.19).

**Kenapa divalidasi di output `transform`, bukan di data mentah?** Output inilah yang masuk ke dashboard dan model. Kolom mentah tetap ada di file ini, sehingga semua aturan tetap bisa dicek, sekaligus memastikan transform tidak merusak data.

| # | Expectation | Kolom | Alasan bisnis |
|---|---|---|---|
| 1 | unique + not null | `borrower_id` | `_id` Elasticsearch; duplikat = dokumen tertimpa |
| 2 | between 18–35 | `age` | Rentang data train model; di luar rentang → prediksi tidak bisa dipercaya |
| 3 | between 0–180 | `days_past_due` | Rentang target valid; prediksi juga di-clip ke rentang ini |
| 4 | in set | `education_level`, `employment_status` | Kategori baru tidak dikenali encoder |
| 5 | > 0 | `monthly_income_idr` | Mencegah pembagian nol di DTI/DSR |
| 6 | between 1–12 | `num_active_lending_apps` | Rentang data train |
| 7 | between 0–1 | `daily_interest_rate_pct` | Bunga harian dalam persen (dataset: 0,10–0,40) |
| 8 | in set | `loan_tenor_months` | Tenor produk {1, 3, 6, 9, 12, 18, 24} |

*Sengaja tidak ditambahkan:* `total_loan_outstanding_idr ≤ credit_limit_granted_idr`. Di dataset, 7,9% debitur memang punya outstanding di atas plafon (utilisasi maks. 1,25). Expectation itu akan gagal padahal datanya memang begitu.

In [1]:
import great_expectations as gx
import pandas as pd
print('GX', gx.__version__)

context = gx.get_context()
datasource = context.sources.add_or_update_pandas(name='finguard_datasource')
asset = datasource.add_csv_asset(name='finguard_transformed', filepath_or_buffer='data/finguard_transformed.csv')
batch_request = asset.build_batch_request()
context.add_or_update_expectation_suite(expectation_suite_name='finguard_suite')
validator = context.get_validator(batch_request=batch_request, expectation_suite_name='finguard_suite')
validator.head()

GX 0.18.19


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,borrower_id,age,monthly_income_idr,regional_minimum_wage_umr_idr,num_active_lending_apps,total_loan_outstanding_idr,credit_limit_granted_idr,loan_tenor_months,daily_interest_rate_pct,gender,...,debt_service_ratio,flag_rate_above_ojk_cap,flag_income_below_3jt,flag_apps_above_3,flag_dsr_above_30,n_pelanggaran,is_macet,kolektibilitas_aktual,dsr_bucket,snapshot_date
0,ID_DEB_00264B68D1,24,6420000,2950000,2,8380000,12280000,1,0.35,Female,...,1.442352,1,0,0,1,2,0,Diragukan,1.0 - 2.0,2026-11-19
1,ID_DEB_0DE3583B12,28,8630000,2950000,1,20110000,29060000,6,0.30,Female,...,0.598096,0,0,0,1,1,0,DPK,0.3 - 1.0,2026-11-19
2,ID_DEB_0F661FADC0,20,5720000,2950000,4,14960000,23900000,1,0.32,Female,...,2.866462,1,0,1,1,3,1,Macet,> 2.0,2026-11-19
3,ID_DEB_10C74D3F21,31,11780000,2950000,3,7620000,26130000,1,0.26,Female,...,0.697314,0,0,0,1,1,0,DPK,0.3 - 1.0,2026-11-19
4,ID_DEB_12CFCAE01F,22,6490000,2950000,2,11080000,16420000,9,0.23,Male,...,0.307493,1,0,0,1,2,0,Lancar,0.3 - 1.0,2026-11-19


In [2]:
# 1. borrower_id unik & tidak kosong
r1a = validator.expect_column_values_to_be_unique('borrower_id')
r1b = validator.expect_column_values_to_not_be_null('borrower_id')
print(r1a.success, r1b.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

True True


In [3]:
# 2. usia 18-35
r2 = validator.expect_column_values_to_be_between('age', min_value=18, max_value=35)
print(r2.success, r2.result['unexpected_count'])

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

True 0


In [4]:
# 3. target days_past_due 0-180 (kolom LLM tidak ada di data pipeline)
r3 = validator.expect_column_values_to_be_between('days_past_due', min_value=0, max_value=180)
print(r3.success, '| kolom LLM tidak ada:', not {'twp90_default_status', 'cot_financial_psychology_rationale'} & set(validator.head().columns))

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

True | kolom LLM tidak ada: True


In [5]:
# 4. kategori pendidikan (4) & pekerjaan (6)
r4a = validator.expect_column_values_to_be_in_set(
    'education_level', ['High School / SMK', 'Diploma (D3)', 'Bachelor (S1)', 'Master (S2)'])
r4b = validator.expect_column_values_to_be_in_set(
    'employment_status', ['Formal Private Employee', 'Gig Worker / Ojol', 'Digital Freelancer / Creator',
                          'Micro-Entrepreneur / UMKM', 'Entry-Level / Jobseeker', 'Civil Servant / BUMN'])
print(r4a.success, r4b.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

True True


In [6]:
# 5. penghasilan > 0 (strict_min -> 0 tidak lolos)
r5 = validator.expect_column_values_to_be_between('monthly_income_idr', min_value=0, strict_min=True)
print(r5.success, '| min:', validator.head(n_rows=10000)['monthly_income_idr'].min())

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

True | min: 1500000


In [7]:
# 6. jumlah aplikasi aktif 1-12
r6 = validator.expect_column_values_to_be_between('num_active_lending_apps', min_value=1, max_value=12)
print(r6.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

True


In [8]:
# 7. bunga harian 0-1 (%)
r7 = validator.expect_column_values_to_be_between('daily_interest_rate_pct', min_value=0, max_value=1)
print(r7.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

True


In [9]:
# 8. tenor sesuai produk
r8 = validator.expect_column_values_to_be_in_set('loan_tenor_months', [1, 3, 6, 9, 12, 18, 24])
print(r8.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

True


In [10]:
validator.save_expectation_suite(discard_failed_expectations=False)
checkpoint = context.add_or_update_checkpoint(name='finguard_checkpoint', validator=validator)
result = checkpoint.run()
stats = result.list_validation_results()[0]['statistics']
print('Semua sukses:', result.success)
print(stats)

Calculating Metrics:   0%|          | 0/69 [00:00<?, ?it/s]

Semua sukses: True
{'evaluated_expectations': 10, 'successful_expectations': 10, 'unsuccessful_expectations': 0, 'success_percent': 100.0}


**Kesimpulan:** semua expectation di atas harus `True`. Jika salah satu gagal, task `validate` di DAG (yang menjalankan aturan yang sama dengan pandas) akan `raise`, sehingga data tidak sampai ke dashboard.

*Kenapa aturan ditulis dua kali (pandas di DAG dan GX di sini)?* Container Airflow DEBlitz belum tentu memasang GX 0.18.19. Cek pandas di DAG adalah penjaga otomatis harian, sedangkan notebook GX adalah dokumentasi kualitas data sesuai kebutuhan validasi.